In [29]:
import torch
import torch.nn as nn

class FeatureSetClassifier(nn.Module):
    def __init__(self, num_features, embed_dim=8):
        super().__init__()

        self.feature_embedding = nn.Embedding(num_features, embed_dim)

        self.phi = nn.Sequential(
            nn.Linear(embed_dim + 1, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # 🔥 THIS is the fix
        self.rho = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, feature_ids, values):
        emb = self.feature_embedding(feature_ids)
        values = values.unsqueeze(1)

        x = torch.cat([emb, values], dim=1)
        z = self.phi(x).mean(dim=0)

        return self.rho(z)

In [39]:
FEATURES = {
    "height": 0,
    "weight": 1,
    "age": 2,   # added later
}

In [24]:
import random

def make_sample(height, weight):
    feature_ids = torch.tensor([0, 1])

    values = torch.tensor([
        height / 200.0,
        weight / 150.0
    ], dtype=torch.float32)

    bmi = weight / ((height / 100) ** 2)
    healthy = 1.0 if 18.5 <= bmi <= 25 else 0.0

    return feature_ids, values, torch.tensor([healthy])

In [30]:
model = FeatureSetClassifier(num_features=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(3000):
    optimizer.zero_grad()
    losses = []

    # batch of samples
    for _ in range(16):
        h = random.uniform(140, 200)
        w = random.uniform(40, 120)

        f_ids, vals, y = make_sample(h, w)
        logits = model(f_ids, vals)

        loss = loss_fn(logits, y)
        losses.append(loss)

    torch.stack(losses).mean().backward()
    optimizer.step()

print("Training complete.")

Training complete.


In [31]:
with torch.no_grad():
    f_ids, vals, _ = make_sample(180, 75)
    print("Healthy (normal):",
          torch.sigmoid(model(f_ids, vals)).item())

    f_ids, vals, _ = make_sample(10, 75)
    print("Healthy (height=10cm):",
          torch.sigmoid(model(f_ids, vals)).item())

Healthy (normal): 0.9322258234024048
Healthy (height=10cm): 8.900234607471524e-14


In [33]:
# 3. Save only the model's state_dict
PATH = "base_model.pt" # Common file extensions are .pt or .pth
torch.save(model.state_dict(), PATH)

In [43]:
old_model = FeatureSetClassifier(num_features=2)
old_model.load_state_dict(torch.load("base_model.pt"))
old_model.eval()

FeatureSetClassifier(
  (feature_embedding): Embedding(2, 8)
  (phi): Sequential(
    (0): Linear(in_features=9, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (rho): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [44]:
new_model = FeatureSetClassifier(num_features=3)  # height, weight, age

In [45]:
# Copy φ and ρ completely
new_model.phi.load_state_dict(old_model.phi.state_dict())
new_model.rho.load_state_dict(old_model.rho.state_dict())

# Copy old embeddings
with torch.no_grad():
    new_model.feature_embedding.weight[:2] = \
        old_model.feature_embedding.weight

In [62]:
# Unfreeze the embedding matrix
new_model.feature_embedding.weight.requires_grad = True

# 🔥 FIX 1: Add a hook to zero-out gradients for the old features (height, weight)
def freeze_old_embeddings(grad):
    mask = torch.ones_like(grad)
    mask[:2, :] = 0.0  # Zero out gradients for index 0 and 1
    return grad * mask

new_model.feature_embedding.weight.register_hook(freeze_old_embeddings)

In [55]:
def make_sample_with_age(height, weight, age):
    feature_ids = torch.tensor([0, 1, 2])
    values = torch.tensor([
        height / 200.0,
        weight / 150.0,
        age / 100.0
    ], dtype=torch.float32)

    bmi = weight / ((height / 100) ** 2)
    healthy = 1.0 if (18.5 <= bmi <= 25 and age < 65) else 0.0

    return feature_ids, values, torch.tensor([healthy])

In [63]:
# 🔥 FIX 2: Pass all unfrozen parameters to the optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, new_model.parameters()), 
    lr=0.01
)

In [64]:
loss_fn = torch.nn.BCEWithLogitsLoss()
import random

# Training loop
for epoch in range(1500):
    optimizer.zero_grad()
    losses = []

    for _ in range(16):
        h = random.uniform(150, 190)
        w = random.uniform(50, 100)
        a = random.uniform(18, 90)

        f_ids, vals, y = make_sample_with_age(h, w, a)
        logits = new_model(f_ids, vals)
        losses.append(loss_fn(logits, y))

    torch.stack(losses).mean().backward()
    optimizer.step()

In [65]:
with torch.no_grad():
    f_ids, vals, _ = make_sample_with_age(180, 75, 25)
    young = torch.sigmoid(new_model(f_ids, vals)).item()

    f_ids, vals, _ = make_sample_with_age(180, 75, 75)
    old = torch.sigmoid(new_model(f_ids, vals)).item()

    print("Healthy (age=25):", young)
    print("Healthy (age=75):", old)

Healthy (age=25): 0.22177621722221375
Healthy (age=75): 0.21705055236816406


In [66]:
# ---------------------------------------------------------
# 1. Copy embeddings and WARM-START the new age embedding
# ---------------------------------------------------------
with torch.no_grad():
    new_model.feature_embedding.weight[:2] = old_model.feature_embedding.weight 
    
    # 🔥 FIX 1: Initialize the new 'age' embedding to the mean of the known embeddings.
    # This guarantees that 'phi' stays in its active ReLU region and preserves 
    # the scale of the mean() pooling layer so 'rho' doesn't die.
    new_model.feature_embedding.weight[2] = old_model.feature_embedding.weight.mean(dim=0)

# Freeze everything first
for p in new_model.parameters():
    p.requires_grad = False

# Unfreeze age embedding
new_model.feature_embedding.weight.requires_grad = True

# Add backward hook to ensure height/weight embeddings don't shift
def freeze_old_embeddings(grad):
    mask = torch.ones_like(grad)
    mask[:2, :] = 0.0  
    return grad * mask

new_model.feature_embedding.weight.register_hook(freeze_old_embeddings)

# ---------------------------------------------------------
# 2. Unfreeze ALL of rho (Highly Recommended)
# ---------------------------------------------------------
# 🔥 FIX 2: While you tried to unfreeze ONLY rho[-1], you are asking the network 
# to learn a new logical AND condition through a frozen bottleneck. Unfreezing 
# all of `rho` allows it to adapt to the new 3-feature mean scale gracefully.
for p in new_model.rho.parameters():
    p.requires_grad = True

# ---------------------------------------------------------
# 3. Give the optimizer ALL unfrozen parameters
# ---------------------------------------------------------
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, new_model.parameters()),
    lr=0.01
)

loss_fn = torch.nn.BCEWithLogitsLoss()
import random

# --- Training loop ---
for epoch in range(1500):
    optimizer.zero_grad()
    losses = []

    for _ in range(16):
        h = random.uniform(150, 190)
        w = random.uniform(50, 100)
        a = random.uniform(18, 90)

        f_ids, vals, y = make_sample_with_age(h, w, a)
        logits = new_model(f_ids, vals)
        losses.append(loss_fn(logits, y))

    torch.stack(losses).mean().backward()
    optimizer.step()

# --- Evaluation ---
print("Age integrated without retraining φ.")
with torch.no_grad():
    f_ids, vals, _ = make_sample_with_age(180, 75, 25)
    young = torch.sigmoid(new_model(f_ids, vals)).item()

    f_ids, vals, _ = make_sample_with_age(180, 75, 75)
    old = torch.sigmoid(new_model(f_ids, vals)).item()

    print("Healthy (age=25):", young)
    print("Healthy (age=75):", old)

Age integrated without retraining φ.
Healthy (age=25): 0.4320920705795288
Healthy (age=75): 0.08798198401927948


In [ ]:
import torch
import torch.nn as nn
import random

class WideAndDeepFeatureSet(nn.Module):
    def __init__(self, max_features=10, embed_dim=16):
        super().__init__()
        
        # --- 1. FEATURE TOKENIZER ---
        # Projects a scalar value into a dense vector space specific to that feature ID
        self.feat_weight = nn.Embedding(max_features, embed_dim)
        self.feat_bias = nn.Embedding(max_features, embed_dim)
        
        # --- 2. WIDE PATH (Additive Bypass) ---
        # Allows new features to push the final logit directly (Zero catastrophic forgetting)
        self.wide_weight = nn.Embedding(max_features, 1)
        self.wide_bias = nn.Embedding(max_features, 1)
        
        # --- 3. DEEP PATH (Feature Interactions) ---
        self.phi = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.LayerNorm(64), # Stabilizes varying sum magnitudes
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        
        self.rho = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, feature_ids, values):
        v = values.unsqueeze(-1) # [batch_size, num_features, 1]
        
        # Token shape: [batch_size, num_features, embed_dim]
        tokens = self.feat_weight(feature_ids) * v + self.feat_bias(feature_ids)
        
        # Deep Path (Interactions like BMI)
        x = self.phi(tokens)
        
        # 🔥 CRITICAL FIX: SUM pooling instead of MEAN.
        z = x.sum(dim=1) 
        deep_logits = self.rho(z) 
        
        # Wide Path (Direct Additive Logic)
        wide_terms = self.wide_weight(feature_ids) * v + self.wide_bias(feature_ids)
        wide_logits = wide_terms.sum(dim=1) 
        
        return deep_logits + wide_logits

# ==========================================
# 1. TRAIN BASE MODEL (Height, Weight only)
# ==========================================
model = WideAndDeepFeatureSet(max_features=10)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Pos_weight = 5.0 handles the class imbalance (Healthy vs Unhealthy logic)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]))

def make_sample(h, w, age=None):
    if age is None:
        f_ids = torch.tensor([0, 1])
        vals = torch.tensor([h / 200.0, w / 150.0], dtype=torch.float32)
    else:
        f_ids = torch.tensor([0, 1, 2])
        vals = torch.tensor([h / 200.0, w / 150.0, age / 100.0], dtype=torch.float32)

    bmi = w / ((h / 100) ** 2)
    healthy = 1.0 if (18.5 <= bmi <= 25) else 0.0
    if age is not None and age >= 65: 
        healthy = 0.0
        
    return f_ids, vals, torch.tensor([healthy])

print("Training Base Model (BMI logic)...")
for epoch in range(800):
    optimizer.zero_grad()
    losses = []
    for _ in range(64):
        h, w = random.uniform(150, 190), random.uniform(50, 100)
        f_ids, vals, y = make_sample(h, w)
        losses.append(loss_fn(model(f_ids.unsqueeze(0), vals.unsqueeze(0)), y.unsqueeze(0)))
    torch.stack(losses).mean().backward()
    optimizer.step()

# ==========================================
# 2. ADD NEW FEATURE (Age) WITH ZERO RETRAINING OF OLD LAYERS
# ==========================================
print("\nAdding Age Feature...")

# Freeze EVERYTHING in the network
for p in model.parameters():
    p.requires_grad = False

# Only unfreeze the embeddings that translate Feature IDs
for emb in [model.feat_weight, model.feat_bias, model.wide_weight, model.wide_bias]:
    emb.weight.requires_grad = True
    
    # Backward hook to strictly prevent updates to Height (0) and Weight (1)
    def freeze_old(grad):
        mask = torch.ones_like(grad)
        mask[:2, :] = 0.0 
        return grad * mask
    emb.weight.register_hook(freeze_old)

# 🔥 INITIALIZATION MAGIC: Set new feature embeddings to exactly 0. 
# Because of SUM pooling, this guarantees zero disruption to old logic on Day 1!
with torch.no_grad():
    model.feat_weight.weight[2] = 0.0
    model.feat_bias.weight[2] = 0.0
    model.wide_weight.weight[2] = 0.0
    model.wide_bias.weight[2] = 0.0

# Optimizer only tracks the embeddings now
opt_new = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)

print("Training strictly on the Age embedding...")
for epoch in range(600):
    opt_new.zero_grad()
    losses = []
    for _ in range(64):
        h, w, a = random.uniform(150, 190), random.uniform(50, 100), random.uniform(18, 90)
        f_ids, vals, y = make_sample(h, w, a)
        losses.append(loss_fn(model(f_ids.unsqueeze(0), vals.unsqueeze(0)), y.unsqueeze(0)))
    torch.stack(losses).mean().backward()
    opt_new.step()

# ==========================================
# 3. EVALUATION
# ==========================================
print("\nEvaluation (with fully frozen interactions):")
with torch.no_grad():
    # Good BMI, Young age
    f_ids, vals, _ = make_sample(180, 75, 25)
    young = torch.sigmoid(model(f_ids.unsqueeze(0), vals.unsqueeze(0))).item()

    # Good BMI, Old age
    f_ids, vals, _ = make_sample(180, 75, 75)
    old = torch.sigmoid(model(f_ids.unsqueeze(0), vals.unsqueeze(0))).item()

    print(f"Healthy (age=25): {young:.4f}  <-- Should be High")
    print(f"Healthy (age=75): {old:.4f}  <-- Should be Low")